<a href="https://colab.research.google.com/github/sawicka-byte/GWB43-groundwater-drought-DIPI/blob/main/03_DIPI_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 DIPI construction

This notebook assembles the final Drought Impact and Pressure Index (DIPI) from precomputed component tables.

The workflow:
- loads the processed DIPI component table,
- calculates the final DIPI score at piezometer level,
- aggregates the results to cluster level,
- exports the cluster summary table.

## Inputs and outputs

### Input file
- `data_processed/DIPI_components.csv`

### Output file
- `data_processed/DIPI_cluster_summary.csv`

In [7]:
import pandas as pd
from pathlib import Path

In [8]:
REPO_NAME = "GWB43-groundwater-drought-DIPI"
REPO_URL = "https://github.com/sawicka-byte/GWB43-groundwater-drought-DIPI.git"

repo_path = Path(f"/content/{REPO_NAME}")

if not repo_path.exists():
    !git clone {REPO_URL}

repo_root = repo_path
print("Repository root:", repo_root)

Repository root: /content/GWB43-groundwater-drought-DIPI


In [9]:
input_file = repo_root / "data_processed" / "DIPI_components.csv"
output_file = repo_root / "data_processed" / "DIPI_cluster_summary.csv"

print("Input:", input_file)
print("Output:", output_file)

Input: /content/GWB43-groundwater-drought-DIPI/data_processed/DIPI_components.csv
Output: /content/GWB43-groundwater-drought-DIPI/data_processed/DIPI_cluster_summary.csv


In [10]:
df = pd.read_csv(input_file)

print("Input shape:", df.shape)
display(df.head())

Input shape: (16, 16)


,piezometer_id,cluster_id,aquifer_group,x_coord_2180,y_coord_2180,exposure,vul_norm,crs_inv_norm,max_dur_norm,pressure,qdens_norm,mining_norm,sensitivity,quality_norm,gde_norm,dipi
0,II/1271/1,1,Shallow unconfined aquifer,441727.379465,523964.377980,0.140790,0.202334,0.103532,0.109647,0.097967,0.335601,0.0,0.342504,0.297427,0.388889,0.188457
1,II/1272/1,1,Shallow unconfined aquifer,406124.320712,559613.681590,0.221605,0.116019,0.382113,0.148469,0.108374,0.401673,0.0,0.148704,0.335471,0.000000,0.165765
2,II/1273/1,1,Shallow unconfined aquifer,457116.257396,519137.508988,0.113071,0.342791,0.120860,0.184122,0.083820,0.448986,0.0,0.379346,0.113432,0.000000,0.184178
3,II/1274/1,1,Shallow unconfined aquifer,437254.525319,574337.265338,0.233976,0.384278,0.941378,0.504456,0.149577,0.407214,0.0,0.117012,0.161228,0.000000,0.173567
4,II/1274/2,1,Shallow unconfined aquifer,437254.525319,574337.265338,0.233976,0.178386,0.444637,0.157965,0.149577,0.430246,0.0,0.117012,0.196089,0.000000,0.173567


In [11]:
print(df.columns.tolist())

['piezometer_id', 'cluster_id', 'aquifer_group', 'x_coord_2180', 'y_coord_2180', 'exposure', 'vul_norm', 'crs_inv_norm', 'max_dur_norm', 'pressure', 'qdens_norm', 'mining_norm', 'sensitivity', 'quality_norm', 'gde_norm', 'dipi']


## Final DIPI at piezometer level

The final DIPI score is calculated here as the arithmetic mean of the three main components already prepared in the input table:
- exposure
- pressure
- sensitivity

In [12]:
df["DIPI"] = (
    df["exposure"] +
    df["pressure"] +
    df["sensitivity"]
) / 3

display(df[[
    "piezometer_id",
    "cluster_id",
    "aquifer_group",
    "exposure",
    "pressure",
    "sensitivity",
    "DIPI"
]].head())

,piezometer_id,cluster_id,aquifer_group,exposure,pressure,sensitivity,DIPI
0,II/1271/1,1,Shallow unconfined aquifer,0.140790,0.097967,0.342504,0.193754
1,II/1272/1,1,Shallow unconfined aquifer,0.221605,0.108374,0.148704,0.159561
2,II/1273/1,1,Shallow unconfined aquifer,0.113071,0.083820,0.379346,0.192079
3,II/1274/1,1,Shallow unconfined aquifer,0.233976,0.149577,0.117012,0.166855
4,II/1274/2,1,Shallow unconfined aquifer,0.233976,0.149577,0.117012,0.166855


## Aggregate DIPI to cluster level

In [13]:
cluster_dipi = (
    df.groupby(["cluster_id", "aquifer_group"], as_index=False)
    .agg({
        "exposure": "mean",
        "pressure": "mean",
        "sensitivity": "mean",
        "DIPI": "mean"
    })
)

cluster_dipi = cluster_dipi.sort_values("cluster_id").reset_index(drop=True)

print("Cluster summary shape:", cluster_dipi.shape)
display(cluster_dipi)

Cluster summary shape: (3, 6)


,cluster_id,aquifer_group,exposure,pressure,sensitivity,DIPI
0,1,Shallow unconfined aquifer,0.225753,0.135852,0.224203,0.195269
1,2,Intermediate confined aquifer,0.317777,0.170616,0.167652,0.218682
2,3,Deep confined aquifer,0.629724,0.430496,0.161141,0.407120


## Export cluster summary

In [14]:
cluster_dipi.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: /content/GWB43-groundwater-drought-DIPI/data_processed/DIPI_cluster_summary.csv


## Reproducibility note

This notebook assembles the final DIPI score from precomputed component tables and provides a cluster-level summary used for interpretation and reporting.